In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from statsmodels.stats.contingency_tables import mcnemar
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load test data
test_df = pd.read_csv('C:/Users/Riyad/projects/fake_news/test_final.csv')
test_df['text'] = test_df['headline'].fillna('') + ' ' + test_df['content'].fillna('')
label2id = {'authentic': 0, 'fake': 1, 'ai_fake': 2}
id2label = {0: 'authentic', 1: 'fake', 2: 'ai_fake'}
test_df['label_id'] = test_df['label'].map(label2id)

print("Libraries & Data loaded!")
print(f"Test: {len(test_df)}")

C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\Riyad\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries & Data loaded!
Test: 2250


In [3]:
# SVM predictions
svm_model = joblib.load('C:/Users/Riyad/projects/fake_news/svm_model.pkl')
svm_preds = svm_model.predict(test_df['text'].values)
svm_pred_ids = np.array([label2id[p] for p in svm_preds])

print("SVM predictions done!")

# Dataset class
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# BanglaBERT predictions
tokenizer = AutoTokenizer.from_pretrained("csebuetnlp/banglabert")
model = AutoModelForSequenceClassification.from_pretrained(
    "csebuetnlp/banglabert", num_labels=3)
model.load_state_dict(torch.load(
    'C:/Users/Riyad/projects/fake_news/banglabert_best.pt'))
model = model.to(device)
model.eval()

test_dataset = NewsDataset(
    test_df['text'].values,
    test_df['label_id'].values,
    tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16)

bert_preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = outputs.logits.argmax(dim=1)
        bert_preds.extend(preds.cpu().numpy())

bert_preds = np.array(bert_preds)
print("BanglaBERT predictions done!")

SVM predictions done!


Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at csebuetnlp/banglabert and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BanglaBERT predictions done!


In [4]:
# McNemar's Test - SVM vs BanglaBERT
true_labels = test_df['label_id'].values

# Correct/Wrong array 
svm_correct = (svm_pred_ids == true_labels)
bert_correct = (bert_preds == true_labels)

# Contingency table


a = np.sum(svm_correct & bert_correct)
b = np.sum(svm_correct & ~bert_correct)
c = np.sum(~svm_correct & bert_correct)
d = np.sum(~svm_correct & ~bert_correct)

print("Contingency Table:")
print(f"Both correct:        {a}")
print(f"SVM only correct:    {b}")
print(f"BERT only correct:   {c}")
print(f"Both wrong:          {d}")

# McNemar's Test
table = [[a, b], [c, d]]
result = mcnemar(table, exact=False, correction=True)

print(f"\n=== McNemar's Test: SVM vs BanglaBERT ===")
print(f"Chi-square: {result.statistic:.4f}")
print(f"P-value:    {result.pvalue:.4f}")

if result.pvalue < 0.05:
    print("Result: Significant difference!  (p < 0.05)")
else:
    print("Result: No significant difference! (p > 0.05)")

Contingency Table:
Both correct:        1995
SVM only correct:    75
BERT only correct:   69
Both wrong:          111

=== McNemar's Test: SVM vs BanglaBERT ===
Chi-square: 0.1736
P-value:    0.6769
Result: No significant difference! (p > 0.05)


In [6]:

print("=== All Models Statistical Summary ===")
print(f"{'Model':<20} {'F1':>8} {'Correct':>10} {'Wrong':>8}")
print("-"*50)

models_results = [
    ("SVM", svm_pred_ids),
    ("BanglaBERT", bert_preds),
]

for name, preds in models_results:
    correct = np.sum(preds == true_labels)
    wrong = np.sum(preds != true_labels)
    f1 = f1_score(true_labels, preds, average='macro')
    print(f"{name:<20} {f1*100:>7.2f}% {correct:>10} {wrong:>8}")

# Save results
import json

stats_results = {
    "mcnemar_svm_vs_bert": {
        "chi_square": float(result.statistic),
        "p_value": float(result.pvalue),
        "significant": bool(result.pvalue < 0.05)
    }
}

with open('C:/Users/Riyad/projects/fake_news/statistical_tests.json', 'w') as f:
    json.dump(stats_results, f)

print(" Statistical tests saved!")

=== All Models Statistical Summary ===
Model                      F1    Correct    Wrong
--------------------------------------------------
SVM                    92.00%       2070      180
BanglaBERT             91.72%       2064      186
 Statistical tests saved!
